1. Importing Required Libraries

In [1]:
import pandas as pd 
import sqlite3 
import json

2. Customer Data

In [2]:
customers = pd.DataFrame({ 
    "CustomerID":[
         "C001", "C002", "C003", "C004", "C004", "C005", "C006", "C007", "C008", "C009", "C010"
         ],
      "CustomerName":[
         "Amina", "Brian", "Carol", "David", "David", "Esther", "Faith", "George", "Halima", "Ian", "Jane" 
         ], 
         "Segment": [ 
            "Retail", "Corporate", "Retail", "SME", "SME", "Corporate", "SME", "Retail", "Corporate", "Retail", "SME" 
            ], 
            "RegionCode": [ 
                "NRB", "MSA", "NRB", "KSM", "KSM", "NKR", "NRB", "MSA", "KSM", "NKR", "NRB" 
            ] 
            }) 
customers

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM
5,C005,Esther,Corporate,NKR
6,C006,Faith,SME,NRB
7,C007,George,Retail,MSA
8,C008,Halima,Corporate,KSM
9,C009,Ian,Retail,NKR


In [3]:
# Saving customer data into csv
customers.to_csv( "customers.csv", index=False )

3. Create the Orders Data

In [4]:
orders = pd.DataFrame({
    "OrderID": [ 
        "O001", "O002", "O003", "O004", "O005", "O006", "O007", "O008", "O009", "O010", "O011", "O012", "O013", "O014", "O015" 
        ], 
        "CustomerID": [ 
        "C001", "C002", "C001", "C003", "C004", "C005", "C006", "C002", "C007", "C008", "C009", "C010", "C006", "C003", "C999" 
        ], 
        "OrderAmount": [
       4500, 12500, 3200, 6800, 9100, 15000, 7600, 8300, 2500, 11000, 5400, 6200, 4300, 7200, 9900 
       ], 
       "ProductCategory": [ 
           "Electronics", "Office", "Home", "Electronics", "Office", "Electronics", "Home", "Office", "Home", "Electronics", "Office", "Home", "Electronics", "Office", "Electronics" 
           ]
            })
orders



,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office
5,O006,C005,15000,Electronics
6,O007,C006,7600,Home
7,O008,C002,8300,Office
8,O009,C007,2500,Home
9,O010,C008,11000,Electronics


In [5]:
# Saving order details in csv
orders.to_csv( "orders.csv", index=False )

4. Create the JSON Region Lookup

In [6]:
regions = [ {"RegionCode": "NRB", "RegionName": "Nairobi"}, {"RegionCode": "MSA", "RegionName": "Mombasa"}, {"RegionCode": "KSM", "RegionName": "Kisumu"}, {"RegionCode": "NKR", "RegionName": "Nakuru"} ]
with open("regions.json", "w") as file: json.dump(regions, file, indent=4)

Part B – Inspect the Sources

In [7]:
customers = pd.read_csv("customers.csv") 
orders = pd.read_csv("orders.csv")

In [8]:
customers.head() 

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


In [9]:
orders.head()

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office


In [10]:
customers.shape

(11, 4)

In [11]:
orders.shape

(15, 4)

Q1. What does one row represent in customers.csv?
One row in customers.csv represents an individual customer record.

Q2. What does one row represent in orders.csv?
One row in orders.csv represents a single order transaction placed by a customer.

Q3. Are the two datasets at the same aggregation grain? Explain briefly.
No, the datasets are at different grains. The customer table is at the individual customer level, while the order table is at the transaction level (meaning one customer can have multiple orders).


Part C – Detect Key Problems

1. Check Duplicates

In [12]:
customers["CustomerID"].duplicated().sum()

1

In [13]:
customers[
 customers["CustomerID"].duplicated(keep=False)
]


,CustomerID,CustomerName,Segment,RegionCode
3,C004,David,SME,KSM
4,C004,David,SME,KSM


Q4. Why can a duplicate CustomerID be dangerous when joining customer and order tables?

A duplicate CustomerID causes rows to multiply during a join, which incorrectly duplicates transactions and artificially inflates row counts and sales totals.

Step 2 – Check Unmatched Customer IDs


In [14]:
set(orders["CustomerID"]) - set(customers["CustomerID"])

{'C999'}

Q5. Which CustomerID appears in orders but not in customers?
C999 appears in the orders table but not in the customers table.

Q6. Why should an unmatched record be investigated rather than silently deleted?
Unmatched records should be investigated because silently deleting them removes valid transaction revenue from the total sales, leading to inaccurate financial reporting.


5. Part D – Create the SQLite Database

In [15]:
conn = sqlite3.connect("retail_lab.db")


In [16]:
customers.to_sql(
 "customers",
 conn,
 if_exists="replace",
 index=False
)
orders.to_sql(
 "orders",
 conn,
 if_exists="replace",
 index=False
)


15

Testing the database

In [17]:
query = """
SELECT *
FROM customers
LIMIT 5;
"""
pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


6. Part E – SELECT and WHERE

Task E1 – Basic SELECT


Query that returns only OrderID, CustomerID and OrderAmount

In [18]:
query = """
SELECT OrderID, CustomerID, OrderAmount
FROM orders;
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount
0,O001,C001,4500
1,O002,C002,12500
2,O003,C001,3200
3,O004,C003,6800
4,O005,C004,9100
5,O006,C005,15000
6,O007,C006,7600
7,O008,C002,8300
8,O009,C007,2500
9,O010,C008,11000


Task E2 – Filter with WHERE


In [19]:
query = """
SELECT * 
FROM orders;
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office
5,O006,C005,15000,Electronics
6,O007,C006,7600,Home
7,O008,C002,8300,Office
8,O009,C007,2500,Home
9,O010,C008,11000,Electronics


Write a query that returns only customers in the Retail segment.

In [20]:
query = """
SELECT * 
FROM customers 
WHERE Segment = 'Retail';
"""
pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C003,Carol,Retail,NRB
2,C007,George,Retail,MSA
3,C009,Ian,Retail,NKR


Write a query that returns orders with OrderAmount greater than KSh 8,000.

In [21]:
query = """
SELECT * 
FROM orders 
WHERE OrderAmount > 8000;
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O002,C002,12500,Office
1,O005,C004,9100,Office
2,O006,C005,15000,Electronics
3,O008,C002,8300,Office
4,O010,C008,11000,Electronics
5,O015,C999,9900,Electronics


Write a query that returns only Electronics orders.

In [22]:
query = """
SELECT * 
FROM orders 
WHERE ProductCategory = 'Electronics';
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O004,C003,6800,Electronics
2,O006,C005,15000,Electronics
3,O010,C008,11000,Electronics
4,O013,C006,4300,Electronics
5,O015,C999,9900,Electronics


Write a query that returns Electronics orders worth more than KSh 8,000.


In [23]:
query = """
SELECT * 
FROM orders 
WHERE ProductCategory = 'Electronics' AND OrderAmount > 8000;
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O006,C005,15000,Electronics
1,O010,C008,11000,Electronics
2,O015,C999,9900,Electronics


7. Part F – GROUP BY and Aggregate Functions

Step 1 – Total Sales

In [24]:
query = """
SELECT 
    SUM(OrderAmount) AS TotalSales
FROM orders;
"""
pd.read_sql_query(query, conn)

,TotalSales
0,113500


Step 2 – Sales by Product Category

In [25]:
query = """
SELECT 
    ProductCategory, 
    SUM(OrderAmount) AS TotalSales
FROM orders
GROUP BY ProductCategory;
"""
pd.read_sql_query(query, conn)

,ProductCategory,TotalSales
0,Electronics,51500
1,Home,19500
2,Office,42500


Task

In [26]:
query = """
SELECT 
    ProductCategory, 
    COUNT(OrderID) AS NumberOfOrders, 
    SUM(OrderAmount) AS TotalSales, 
    AVG(OrderAmount) AS AverageOrderValue
FROM orders
GROUP BY ProductCategory;
"""
pd.read_sql_query(query, conn)

,ProductCategory,NumberOfOrders,TotalSales,AverageOrderValue
0,Electronics,6,51500,8583.333333
1,Home,4,19500,4875.000000
2,Office,5,42500,8500.000000


Q7. Write two short observations about category performance. Refer to actual values from your output.

Electronics is the highest-performing category in terms of volume and revenue, leading with 6 orders and reaching 51,500 in total sales.

The Home category lags significantly behind the others in transaction size, generating the lowest average order value at 4,875, which is roughly half the average order value of Electronics (8,583.3333) and Office (8,500).

8. Part G – JOIN Customers and Orders

In [27]:
query = """
SELECT
    o.OrderID,
    o.CustomerID,
    c.CustomerName,
    c.Segment,
    o.OrderAmount
FROM orders o
LEFT JOIN customers c
    ON o.CustomerID = c.CustomerID;
"""
joined = pd.read_sql_query(query, conn)
joined

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O005,C004,David,SME,9100
6,O006,C005,Esther,Corporate,15000
7,O007,C006,Faith,SME,7600
8,O008,C002,Brian,Corporate,8300
9,O009,C007,George,Retail,2500


In [28]:
len(orders)


15

In [29]:
len(joined)

16

Q8. Are the row counts equal? If not, explain exactly what caused the difference.

No, the row counts are not equal (len(orders) is 15, while len(joined) is 16). The difference was caused by CustomerID = C004 appearing twice in the customers table. Because C004 has two rows in customers, the LEFT JOIN matched Order 0005 twice, creating a duplicate order row in the joined output

Q9. Locate the unmatched customer record in the joined output. What values appear for CustomerName and
Segment?
The unmatched customer record is OrderID = 0015 with CustomerID = C999. For this row, both CustomerName and Segment appear as None because C999 does not exist in the customers table.

9. Part H – Reconcile the Totals


Before the JOIN


In [30]:
orders["OrderAmount"].sum()


113500

After the JOIN

In [31]:
joined["OrderAmount"].sum()


122600

Q10. Are the two sales totals equal?

No, the sales totals are not equal. orders["OrderAmount"].sum() is 113,500, whereas joined["OrderAmount"].sum() is 122,600 an  increase of 9,100.

Q11. If the total changed, which customer caused the duplication?
Customer C004 -David caused the duplication because duplicate entries for C004 in the customers table caused Order 0005 with amount 9,100 to be counted twice.

Q12. Why is total reconciliation an important validation step after a JOIN?

Total reconciliation acts as a control check to catch duplicate keys, fan-out issues, or grain mismatches created during a join. If total sales change after joining customer attributes, it alerts analysts that data integrity has been compromised before publishing financial KPIs.

10. Part I – Correct the Duplicate-Key Problem

In [32]:
customers_clean = (
 customers
 .drop_duplicates(subset="CustomerID")
)


In [33]:
customers_clean.to_sql(
    "customers_clean",
    conn,
    if_exists="replace",
    index=False
)

10

In [34]:
query = """
SELECT
    o.OrderID,
    o.CustomerID,
    c.CustomerName,
    c.Segment,
    o.OrderAmount
FROM orders o
LEFT JOIN customers_clean c
    ON o.CustomerID = c.CustomerID;
"""
clean_joined = pd.read_sql_query(query, conn)
clean_joined

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O006,C005,Esther,Corporate,15000
6,O007,C006,Faith,SME,7600
7,O008,C002,Brian,Corporate,8300
8,O009,C007,George,Retail,2500
9,O010,C008,Halima,Corporate,11000


In [35]:
len(orders), len(clean_joined)

(15, 15)

In [36]:
orders["OrderAmount"].sum(), clean_joined["OrderAmount"].sum()

(113500, 113500)

Q13. Has the corrected JOIN preserved both the row count and the total sales value? Explain.

Yes, both row counts are now equal (15 rows each), and both sales totals are equal (113,500). Removing the duplicate C004 record restored a unique 1-to-1 matching key in customers_clean, preventing order amounts from duplicating during the LEFT JOIN.


11. Part J – Customer Segment KPIs
    

In [37]:
query = """
SELECT 
    COALESCE(c.Segment, 'Unmatched / NULL') AS Segment,
    COUNT(o.OrderID) AS Orders,
    SUM(o.OrderAmount) AS TotalSales,
    AVG(o.OrderAmount) AS AverageOrderValue
FROM orders o
LEFT JOIN customers_clean c
    ON o.CustomerID = c.CustomerID
GROUP BY c.Segment;
"""
pd.read_sql_query(query, conn)

,Segment,Orders,TotalSales,AverageOrderValue
0,Unmatched / NULL,1,9900,9900.000000
1,Corporate,4,46800,11700.000000
2,Retail,6,29600,4933.333333
3,SME,4,27200,6800.000000


Q14. Write three sentences explaining what management can learn from the segment KPIs

Corporate customers are the primary revenue driver (46,800 in total sales) and yield the highest average order value (11,700), making them the most profitable segment per transaction.

Retail generates the highest order volume with 6 total orders, but produces the lowest average order value (4,933.33), indicating high-frequency but smaller purchases.

An unmatched transaction (C999) accounts for 9,900 in unallocated sales, highlighting a data acquisition issue that management needs to clean up to properly attribute revenue.

12. Part K – Merge the JSON Region Lookup


In [38]:
regions_df = pd.read_json("regions.json")
regions_df
customers_regions = customers_clean.merge(
 regions_df,
 on="RegionCode",
 how="left"
)
customers_regions.head()


,CustomerID,CustomerName,Segment,RegionCode,RegionName
0,C001,Amina,Retail,NRB,Nairobi
1,C002,Brian,Corporate,MSA,Mombasa
2,C003,Carol,Retail,NRB,Nairobi
3,C004,David,SME,KSM,Kisumu
4,C005,Esther,Corporate,NKR,Nakuru


Q15. Confirm that each customer now has a readable RegionName rather than only a RegionCode.

Inspecting customers_regions.head() confirms that the LEFT JOIN successfully appended the RegionName column (e.g Nairobi, Mombasa, Kisumu, Nakuru) alongside each customer's RegionCode.


Q16. If a RegionCode did not exist in regions.json, what would a left merge produce for RegionName?

left merge preserves all rows from the primary table (customers_clean). If a RegionCode were missing from regions.json, pandas would populate RegionName with None or those unmatched rows

13. Final Challenge – Regional Sales Analysis


In [39]:
customers_regions.to_sql(
    "customers_regions",
    conn,
    if_exists="replace",
    index=False
)

query = """
SELECT 
    COALESCE(cr.RegionName, 'Unmatched / NULL') AS Region,
    COUNT(o.OrderID) AS NumberOfOrders,
    SUM(o.OrderAmount) AS TotalSales
FROM orders o
LEFT JOIN customers_regions cr
    ON o.CustomerID = cr.CustomerID
GROUP BY cr.RegionName;
"""
regional_summary = pd.read_sql_query(query, conn)
regional_summary

,Region,NumberOfOrders,TotalSales
0,Unmatched / NULL,1,9900
1,Kisumu,2,20100
2,Mombasa,3,23300
3,Nairobi,7,39800
4,Nakuru,2,20400


Q17. Which region generated the highest sales?

Nairobi generated the highest sales, producing KSh 39,800 across 7 orders.

Q18. Should management automatically conclude that this is the “best-performing” region?Explain using at
least two limitations of the available data.

No, management should not automatically conclude Nairobi is the best-performing region.

1.Lack of Profitability & Margin Data: The dataset only tracks gross revenue (OrderAmount) and contains no information on cost of goods sold, shipping expenses, or operational overhead. A region with high revenue could have lower net profit margins.

2. Small Sample Size & Duration Bias: With only 15 transactions total, the sample size is extremely small and may reflect short-term noise or a few outlier transactions rather than long-term regional performance.

14. Short Conclusion

Conclusion

The most critical data-quality issue identified was duplicate primary keys (C004) in the customer source file. When joined without deduplication, this non-unique key caused a fan-out effect that artificially inflated total sales from KSh 113,500 to KSh 122,600. Reconciling control totals after joins is essential to ensure relational logic has not corrupted analytical findings. Among verified customer segments, the Corporate segment produced the highest total revenue (KSh 46,800) and the highest average order value (KSh 11,700). A key limitation of this analysis is the small sample size (15 orders) and the absence of cost data required to measure true profitability.